# Sparse Attention (稀疏注意力)

## 概述

稀疏注意力通过只计算部分位置对的注意力来降低计算复杂度。标准注意力需要计算所有n²个位置对，而稀疏注意力通过精心设计的模式只计算有意义的连接。

### 核心思想

**观察**: 并非所有位置对都需要计算注意力
- 距离很远的位置可能关系不大
- 某些特殊位置（如[CLS]）需要全局视野
- 随机连接可以捕获长距离依赖

**解决方案**: 只计算稀疏连接
```
标准注意力: 计算所有 n² 个位置对
稀疏注意力: 只计算 n×k 个位置对 (k << n)
```

### 常见稀疏模式

1. **Local (局部)**: 滑动窗口
2. **Global (全局)**: 特殊token
3. **Random (随机)**: 随机连接
4. **Longformer**: Local + Global
5. **BigBird**: Local + Global + Random

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)

## 1. 稀疏模式生成函数

In [ ]:
def create_local_mask(seq_len, window_size):
    """
    局部注意力：滑动窗口
    每个位置只关注前后window_size范围内的位置
    """
    mask = np.zeros((seq_len, seq_len))
    
    for i in range(seq_len):
        start = max(0, i - window_size)
        end = min(seq_len, i + window_size + 1)
        mask[i, start:end] = 1
    
    return mask


def create_global_mask(seq_len, global_indices):
    """
    全局注意力：特定位置关注所有位置
    """
    mask = np.zeros((seq_len, seq_len))
    
    for idx in global_indices:
        mask[idx, :] = 1  # 全局位置关注所有
        mask[:, idx] = 1  # 所有位置关注全局
    
    return mask


def create_random_mask(seq_len, num_random):
    """
    随机注意力：每个位置随机关注k个位置
    """
    mask = np.zeros((seq_len, seq_len))
    
    for i in range(seq_len):
        random_indices = np.random.choice(seq_len, size=num_random, replace=False)
        mask[i, random_indices] = 1
        mask[i, i] = 1  # 自己总是可见
    
    return mask


def create_longformer_mask(seq_len, window_size, global_indices):
    """
    Longformer: Local + Global
    """
    local_mask = create_local_mask(seq_len, window_size)
    global_mask = create_global_mask(seq_len, global_indices)
    return np.maximum(local_mask, global_mask)


def create_bigbird_mask(seq_len, window_size, num_random, global_indices):
    """
    BigBird: Local + Global + Random
    """
    local_mask = create_local_mask(seq_len, window_size)
    global_mask = create_global_mask(seq_len, global_indices)
    random_mask = create_random_mask(seq_len, num_random)
    return np.maximum(np.maximum(local_mask, global_mask), random_mask)

## 2. 可视化稀疏模式

In [ ]:
# 生成不同的稀疏模式
seq_len = 64

patterns = {
    'Local (窗口=3)': create_local_mask(seq_len, window_size=3),
    'Global (位置0)': create_global_mask(seq_len, [0]),
    'Random (k=5)': create_random_mask(seq_len, num_random=5),
    'Longformer': create_longformer_mask(seq_len, window_size=3, global_indices=[0]),
    'BigBird': create_bigbird_mask(seq_len, window_size=3, num_random=3, global_indices=[0])
}

# 绘制所有模式
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, mask) in enumerate(patterns.items()):
    ax = axes[idx]
    
    # 计算稀疏度
    sparsity = np.sum(mask) / (seq_len * seq_len)
    
    # 绘制mask
    im = ax.imshow(mask, cmap='Blues', interpolation='nearest', aspect='auto')
    ax.set_title(f'{name}\n稀疏度: {sparsity:.1%} ({int(np.sum(mask))}/{seq_len*seq_len} 连接)', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Key位置', fontsize=11)
    ax.set_ylabel('Query位置', fontsize=11)
    
    # 添加colorbar
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# 移除多余的subplot
axes[-1].axis('off')

plt.suptitle('稀疏注意力模式对比', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# 打印统计信息
print("\n稀疏度统计:")
print("-" * 60)
for name, mask in patterns.items():
    sparsity = np.sum(mask) / (seq_len * seq_len)
    print(f"{name:<25} 稀疏度: {sparsity:>6.1%}  连接数: {int(np.sum(mask)):>5}")

## 3. 稀疏注意力实现

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


class SparseAttention:
    """
    稀疏注意力实现
    """
    
    def __init__(self, embed_dim, pattern='local', **kwargs):
        self.embed_dim = embed_dim
        self.pattern = pattern
        self.kwargs = kwargs
        
        # 初始化投影矩阵
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def create_mask(self, seq_len):
        """根据pattern创建mask"""
        if self.pattern == 'local':
            window_size = self.kwargs.get('window_size', 3)
            return create_local_mask(seq_len, window_size)
        elif self.pattern == 'longformer':
            window_size = self.kwargs.get('window_size', 3)
            global_indices = self.kwargs.get('global_indices', [0])
            return create_longformer_mask(seq_len, window_size, global_indices)
        elif self.pattern == 'bigbird':
            window_size = self.kwargs.get('window_size', 3)
            num_random = self.kwargs.get('num_random', 3)
            global_indices = self.kwargs.get('global_indices', [0])
            return create_bigbird_mask(seq_len, window_size, num_random, global_indices)
        else:
            raise ValueError(f"Unknown pattern: {self.pattern}")
    
    def forward(self, x, return_attention=False):
        """
        前向传播
        """
        seq_len, embed_dim = x.shape
        
        # 投影
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        # 创建稀疏mask
        mask = self.create_mask(seq_len)
        
        # 计算注意力得分
        scores = np.dot(Q, K.T) / np.sqrt(embed_dim)
        
        # 应用mask
        scores = np.where(mask == 1, scores, -1e9)
        
        # Softmax
        attention = softmax(scores, axis=-1)
        
        # 加权求和
        output = np.dot(attention, V)
        
        if return_attention:
            return output, attention, mask
        return output


# 测试
embed_dim = 32
seq_len = 64
x = np.random.randn(seq_len, embed_dim)

# 创建不同的稀疏注意力
local_attn = SparseAttention(embed_dim, pattern='local', window_size=5)
output_local, attn_local, mask_local = local_attn.forward(x, return_attention=True)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output_local.shape}")
print(f"注意力权重形状: {attn_local.shape}")
print(f"稀疏度: {np.sum(mask_local)/(seq_len*seq_len):.2%}")

## 4. 注意力权重可视化

In [ ]:
# 对比不同模式的注意力权重
patterns_to_test = [
    ('Local (窗口=5)', {'pattern': 'local', 'window_size': 5}),
    ('Longformer', {'pattern': 'longformer', 'window_size': 5, 'global_indices': [0, seq_len-1]}),
    ('BigBird', {'pattern': 'bigbird', 'window_size': 5, 'num_random': 5, 'global_indices': [0]})
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, config) in enumerate(patterns_to_test):
    sparse_attn = SparseAttention(embed_dim, **config)
    _, attention, mask = sparse_attn.forward(x, return_attention=True)
    
    ax = axes[idx]
    im = ax.imshow(attention, cmap='YlOrRd', interpolation='nearest', aspect='auto')
    ax.set_title(f'{name}\n稀疏度: {np.sum(mask)/(seq_len*seq_len):.1%}', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Key位置', fontsize=11)
    ax.set_ylabel('Query位置', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('稀疏注意力权重对比', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 稀疏度分析

分析不同序列长度下的稀疏度变化。

In [ ]:
# 不同序列长度的稀疏度
seq_lengths = [64, 128, 256, 512, 1024, 2048]
window_size = 5
num_random = 5

results = {
    'Local': [],
    'Longformer': [],
    'BigBird': []
}

for seq_len in seq_lengths:
    # Local
    local_mask = create_local_mask(seq_len, window_size)
    results['Local'].append(np.sum(local_mask) / (seq_len * seq_len))
    
    # Longformer
    lf_mask = create_longformer_mask(seq_len, window_size, [0])
    results['Longformer'].append(np.sum(lf_mask) / (seq_len * seq_len))
    
    # BigBird
    bb_mask = create_bigbird_mask(seq_len, window_size, num_random, [0])
    results['BigBird'].append(np.sum(bb_mask) / (seq_len * seq_len))

# 绘制稀疏度曲线
plt.figure(figsize=(10, 6))
for name, sparsities in results.items():
    plt.plot(seq_lengths, sparsities, 'o-', label=name, linewidth=2, markersize=8)

plt.xlabel('序列长度', fontsize=12)
plt.ylabel('稀疏度', fontsize=12)
plt.title('稀疏度随序列长度的变化\n(窗口大小=5, 随机连接=5)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.show()

# 打印数据
print("\n稀疏度数据:")
print("-" * 70)
print(f"{'序列长度':<12}", end='')
for name in results.keys():
    print(f"{name:<20}", end='')
print()
print("-" * 70)

for i, seq_len in enumerate(seq_lengths):
    print(f"{seq_len:<12}", end='')
    for name in results.keys():
        print(f"{results[name][i]:<20.4%}", end='')
    print()

## 6. 计算复杂度对比

理论复杂度分析。

In [ ]:
# 计算理论FLOPs
def compute_flops(seq_len, embed_dim, sparsity):
    """
    计算注意力机制的理论FLOPs
    
    主要计算:
    1. QK^T: seq_len × seq_len × embed_dim × sparsity
    2. Softmax: seq_len × seq_len × sparsity
    3. Attention @ V: seq_len × seq_len × embed_dim × sparsity
    """
    qk_flops = seq_len * seq_len * embed_dim * sparsity
    softmax_flops = seq_len * seq_len * sparsity
    av_flops = seq_len * seq_len * embed_dim * sparsity
    
    total_flops = qk_flops + softmax_flops + av_flops
    return total_flops


# 不同模式的FLOPs对比
seq_lengths = [128, 256, 512, 1024, 2048]
embed_dim = 64
window_size = 5

flops_data = {
    'Full (标准)': [],
    'Local': [],
    'Longformer': [],
    'BigBird': []
}

for seq_len in seq_lengths:
    # Full attention
    full_flops = compute_flops(seq_len, embed_dim, 1.0)
    flops_data['Full (标准)'].append(full_flops)
    
    # Local
    local_mask = create_local_mask(seq_len, window_size)
    local_sparsity = np.sum(local_mask) / (seq_len * seq_len)
    flops_data['Local'].append(compute_flops(seq_len, embed_dim, local_sparsity))
    
    # Longformer
    lf_mask = create_longformer_mask(seq_len, window_size, [0])
    lf_sparsity = np.sum(lf_mask) / (seq_len * seq_len)
    flops_data['Longformer'].append(compute_flops(seq_len, embed_dim, lf_sparsity))
    
    # BigBird
    bb_mask = create_bigbird_mask(seq_len, window_size, 5, [0])
    bb_sparsity = np.sum(bb_mask) / (seq_len * seq_len)
    flops_data['BigBird'].append(compute_flops(seq_len, embed_dim, bb_sparsity))

# 绘制FLOPs对比
plt.figure(figsize=(12, 6))

for name, flops in flops_data.items():
    # 转换为GFLOPs
    gflops = [f / 1e9 for f in flops]
    marker = 'o' if name == 'Full (标准)' else 's'
    linewidth = 3 if name == 'Full (标准)' else 2
    plt.plot(seq_lengths, gflops, marker=marker, label=name, 
             linewidth=linewidth, markersize=8)

plt.xlabel('序列长度', fontsize=12)
plt.ylabel('理论FLOPs (GFLOPs)', fontsize=12)
plt.title('注意力机制计算复杂度对比', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.xscale('log')
plt.tight_layout()
plt.show()

# 打印加速比
print("\n相对于标准注意力的加速比:")
print("-" * 70)
print(f"{'序列长度':<12}", end='')
for name in ['Local', 'Longformer', 'BigBird']:
    print(f"{name:<20}", end='')
print()
print("-" * 70)

for i, seq_len in enumerate(seq_lengths):
    print(f"{seq_len:<12}", end='')
    full_flops = flops_data['Full (标准)'][i]
    for name in ['Local', 'Longformer', 'BigBird']:
        speedup = full_flops / flops_data[name][i]
        print(f"{speedup:<20.2f}x", end='')
    print()

## 7. 局部注意力的窗口大小影响

In [ ]:
# 不同窗口大小的影响
seq_len = 512
window_sizes = [1, 2, 3, 5, 8, 16, 32, 64]

sparsities = []
for ws in window_sizes:
    mask = create_local_mask(seq_len, ws)
    sparsities.append(np.sum(mask) / (seq_len * seq_len))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 稀疏度曲线
ax1.plot(window_sizes, sparsities, 'o-', linewidth=2, markersize=8, color='steelblue')
ax1.set_xlabel('窗口大小', fontsize=12)
ax1.set_ylabel('稀疏度', fontsize=12)
ax1.set_title(f'局部注意力稀疏度 vs 窗口大小\n(序列长度={seq_len})', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# 活跃连接数
active_connections = [s * seq_len * seq_len for s in sparsities]
ax2.plot(window_sizes, active_connections, 's-', linewidth=2, markersize=8, color='coral')
ax2.set_xlabel('窗口大小', fontsize=12)
ax2.set_ylabel('活跃连接数', fontsize=12)
ax2.set_title(f'活跃连接数 vs 窗口大小\n(序列长度={seq_len})', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print("\n窗口大小影响:")
print("-" * 50)
for ws, sp in zip(window_sizes, sparsities):
    print(f"窗口={ws:<3}  稀疏度={sp:>6.2%}  连接数={int(sp*seq_len*seq_len):>7}")

## 8. 总结

### 稀疏注意力的优势

1. **计算效率**: O(n²) → O(n×k)
2. **内存节省**: 减少存储注意力矩阵的开销
3. **可扩展性**: 能处理更长的序列
4. **灵活性**: 可以设计任务相关的稀疏模式

### 稀疏模式选择指南

| 模式 | 适用场景 | 稀疏度 | 复杂度 |
|------|---------|--------|--------|
| **Local** | 局部相关性强（语言建模） | 低 | O(n×w) |
| **Longformer** | 需要全局token（文档分类） | 中 | O(n×w + n×g) |
| **BigBird** | 平衡局部和全局（通用） | 中高 | O(n×(w+r+g)) |

其中: w=窗口大小, g=全局token数, r=随机连接数

### 实际应用

- **Longformer**: 长文档理解、文档问答
- **BigBird**: 基因组序列、长文本摘要
- **Sparse Transformer**: 图像生成、音乐生成

### 权衡

- ⚠️ 需要精心设计稀疏模式
- ⚠️ 某些任务可能需要密集注意力
- ⚠️ 实现复杂度较高（需要高效的稀疏矩阵操作）